## import / set up

In [19]:
%load_ext autoreload
%autoreload 2

import sys
from datetime import datetime, timedelta
from pathlib import Path
import json

import pandas as pd
import numpy as np

# 將共整合目錄新增到路徑以匯入 get_data 模組
sys.path.insert(0, str(Path.cwd()))

from get_data import (
    create_bybit_session,
    create_bybit_session_from_config,
    get_all_bybit_usdt_spot_symbols,
    get_all_bybit_perp_symbols,
    get_price,
    get_funding_rate,
    fetch_sector_map,
    get_binance_futures_klines,
    get_binance_spot_klines,
    get_binance_funding_rate,
    get_all_binance_spot_symbols,
    get_all_binance_futures_symbols,
    get_binance_spot_klines,
    get_binance_futures_klines,
    get_binance_funding_rate
)

# 配置路徑
DATABASE_PATH = Path("data")
CONFIG_PATH = Path("config/api_config.json")  # 選填：用於 API 認證

# 定義數據收集的日期範圍
START_DATE = datetime(2020, 1, 1)
END_DATE = datetime.now()

print(f"數據收集範圍: {START_DATE.date()} 至 {END_DATE.date()}")
print(f"資料庫路徑: {DATABASE_PATH}")
print("✓ 數據格式: Parquet (高效壓縮和快速查詢)")

# 創建 Bybit 會話（公共端點無需認證即可使用）
# 如果需要使用認證端點，請使用：
# bybit_session = create_bybit_session_from_config(CONFIG_PATH)

try:
    bybit_session = create_bybit_session()
    print("✓ Bybit 會話創建成功")
except ImportError as e:
    print(f"⚠ {e}")
    bybit_session = None

# 獲取所有可用的 USDT 現貨符號
try:
    spot_symbols = get_all_bybit_usdt_spot_symbols()
    print(f"\n✓ 已獲取 {len(spot_symbols)} 個 Bybit 現貨 USDT 符號")
    print(f"  範例符號: {spot_symbols[:5]}")
except Exception as e:
    print(f"⚠ 獲取現貨符號出錯: {e}")
    spot_symbols = []

# 獲取所有可用的 USDT 永續符號
try:
    perp_symbols = get_all_bybit_perp_symbols()
    print(f"\n✓ 已獲取 {len(perp_symbols)} 個 Bybit 永續 USDT 符號")
    print(f"  範例符號: {perp_symbols[:5]}")
except Exception as e:
    print(f"⚠ 獲取永續符號出錯: {e}")
    perp_symbols = []

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
數據收集範圍: 2020-01-01 至 2026-05-26
資料庫路徑: data
✓ 數據格式: Parquet (高效壓縮和快速查詢)
✓ Bybit 會話創建成功

✓ 已獲取 447 個 Bybit 現貨 USDT 符號
  範例符號: ['BTCUSDT', 'ETHUSDT', 'XRPUSDT', 'DOTUSDT', 'XLMUSDT']

✓ 已獲取 570 個 Bybit 永續 USDT 符號
  範例符號: ['0GUSDT', '1000000BABYDOGEUSDT', '1000000CHEEMSUSDT', '1000000MOGUSDT', '10000NEXUSDT']


## Fetch Sector Classification

Use `fetch_sector_map()` to retrieve CoinGecko sector classifications for downloaded symbols. Generate metadata JSON files mapping symbols to sectors.

In [20]:
print("正在從 CoinGecko 獲取行業分類...")
print("這將匹配下載的符號及其區塊鏈行業。\n")

try:
    fetch_sector_map(
        base_path=DATABASE_PATH,
        categories_to_process=("spot", "linear"),
        sleep_seconds=0.5
    )
    print("\n✓ 行業分類映射完成")
    
    # 檢查是否已創建元數據文件
    metadata_dir = DATABASE_PATH / "metadata"
    if metadata_dir.exists():
        sector_files = list(metadata_dir.glob("*_sector_map.json"))
        print(f"✓ 已生成 {len(sector_files)} 個行業映射文件:")
        for f in sector_files:
            print(f"  - {f.name}")
except ImportError as e:
    print(f"⚠ {e}")
except Exception as e:
    print(f"⚠ 獲取行業映射出錯: {e}")

正在從 CoinGecko 獲取行業分類...
這將匹配下載的符號及其區塊鏈行業。

Fetching CoinGecko category list...
Selected Top 10 Categories by Market Cap:
  1. Smart Contract Platform
  2. Layer 1 (L1)
  3. Proof of Work (PoW)
  4. World Liberty Financial Portfolio
  5. Proof of Stake (PoS)
  6. Stablecoins
  7. Made in USA
  8. USD Stablecoin
  9. Fiat-backed Stablecoin
  10. Exchange-based Tokens


Fetching Top 10 sectors: 100%|██████████| 10/10 [02:10<00:00, 13.07s/it]

Successfully saved Top 10 sector map to top10_market_sector_map.json

✓ 行業分類映射完成
✓ 已生成 1 個行業映射文件:
  - top10_market_sector_map.json


## Download Sample Data (Spot & Futures)

First download a sample of symbols to create the data structure, then generate sector classification metadata based on the downloaded symbols.

In [22]:
# 1. 讀取 Top 10 板塊 metadata
metadata_path = DATABASE_PATH / "metadata" / "top10_market_sector_map.json"
with open(metadata_path, "r", encoding="utf-8") as f:
    sector_map = json.load(f)

# ==========================================
# 2. 參數設定：在這裡填入您想跳過的板塊名稱
# ==========================================
skip_sectors = ["USD Stablecoin", "Fiat-backed Stablecoin"] 

# 3. 收集 CoinGecko 上的幣種 (基礎 Universe)
cg_symbols = set()

for sector_name, info in sector_map.items():
    if sector_name in skip_sectors:
        print(f"⏭️ 跳過板塊: {sector_name}")
        continue
        
    cg_symbols.update(info["symbols"])

# 4. 獲取 Binance 分離的 Universe
print("\n獲取 Binance 支援幣種清單中...")
binance_spot_universe = set(get_all_binance_spot_symbols())
binance_futures_universe = set(get_all_binance_futures_symbols())

# 5. 嚴格分離交集
# 現貨清單：必須在 CoinGecko 且在 Binance Spot Universe 中
spot_symbols_to_download = list(cg_symbols & binance_spot_universe)
# 合約清單：必須在 CoinGecko 且在 Binance Futures Universe 中
futures_symbols_to_download = list(cg_symbols & binance_futures_universe)

print("-" * 40)
print(f"CoinGecko 統計總幣種數: {len(cg_symbols)} 個")
print(f"有效現貨 (Spot) 下載數量: {len(spot_symbols_to_download)} 個")
print(f"有效合約 (Futures) 下載數量: {len(futures_symbols_to_download)} 個\n")

# 6. 開始執行下載 Binance 數據
print(f"日期範圍: {START_DATE.date()} 至 {END_DATE.date()}")

try:
    # 📥 [1/3] 下載 Binance 合約 (U本位) 數據
    if futures_symbols_to_download:
        print(f"\n📥 [1/3] 下載 {len(futures_symbols_to_download)} 個 Binance 合約 K 線 (1h)...")
        get_binance_futures_klines(
            symbols=futures_symbols_to_download,
            start_date=START_DATE,
            end_date=END_DATE,
            database=DATABASE_PATH,
            interval="1h",       
            sleep_seconds=0.5    
        )
    
    # 📥 [2/3] 下載 Binance 現貨數據
    if spot_symbols_to_download:
        print(f"📥 [2/3] 下載 {len(spot_symbols_to_download)} 個 Binance 現貨 K 線 (1h)...")
        get_binance_spot_klines(
            symbols=spot_symbols_to_download,
            start_date=START_DATE,
            end_date=END_DATE,
            database=DATABASE_PATH,
            interval="1h",
            sleep_seconds=0.5
        )
    
    # # 📥 [3/3] 下載 Binance 融資費率數據
    # if futures_symbols_to_download:
    #     print(f"📥 [3/3] 下載 {len(futures_symbols_to_download)} 個 Binance 融資費率...")
    #     # 融資費率必定對應合約市場，所以使用 futures_symbols_to_download
    #     get_binance_funding_rate(
    #         symbols=futures_symbols_to_download,
    #         start_date=START_DATE,
    #         end_date=END_DATE,
    #         database=DATABASE_PATH,
    #         sleep_seconds=0.5
    #     )
    
    print("\n✓ Binance 所有數據下載完成！")
    
except Exception as e:
    print(f"⚠ 下載過程中出錯: {e}")

⏭️ 跳過板塊: USD Stablecoin
⏭️ 跳過板塊: Fiat-backed Stablecoin

獲取 Binance 支援幣種清單中...
----------------------------------------
CoinGecko 統計總幣種數: 1119 個
有效現貨 (Spot) 下載數量: 228 個
有效合約 (Futures) 下載數量: 232 個

日期範圍: 2020-01-01 至 2026-05-26

📥 [1/3] 下載 232 個 Binance 合約 K 線 (1h)...
📥 [2/3] 下載 228 個 Binance 現貨 K 線 (1h)...
📥 [3/3] 下載 232 個 Binance 融資費率...

✓ Binance 所有數據下載完成！


## 9. Data Exploration and Validation

Load and inspect downloaded CSV files. Check data completeness, date ranges, and data types. Verify all symbols have valid OHLCV and funding rate data.

In [ ]:
print("=" * 60)
print("數據探索和驗證")
print("=" * 60)

def explore_parquet_files(directory):
    """在目錄中載入和檢查 Parquet 文件。"""
    parquet_files = list(Path(directory).glob("*.parquet"))
    if not parquet_files:
        print(f"未在 {directory} 中找到 Parquet 文件")
        return
    
    print(f"\n在 {directory} 中找到 {len(parquet_files)} 個 Parquet 文件:")
    for i, parquet_file in enumerate(parquet_files[:5], 1):
        try:
            df = pd.read_parquet(parquet_file)
            print(f"\n  {i}. {parquet_file.name}")
            print(f"     形狀: {df.shape[0]} 行 × {df.shape[1]} 列")
            print(f"     列: {list(df.columns)}")
            print(f"     數據類型: {df.dtypes.to_dict()}")
            if isinstance(df.index, pd.DatetimeIndex):
                print(f"     日期範圍: {df.index[0]} 至 {df.index[-1]}")
        except Exception as e:
            print(f"\n  {i}. {parquet_file.name} - 錯誤: {e}")
    
    if len(parquet_files) > 5:
        print(f"\n  ... 還有 {len(parquet_files) - 5} 個文件")

# 探索不同的數據類別
if (DATABASE_PATH / "spot").exists():
    explore_parquet_files(DATABASE_PATH / "spot")

if (DATABASE_PATH / "linear").exists():
    explore_parquet_files(DATABASE_PATH / "linear")

if (DATABASE_PATH / "funding_rate").exists():
    explore_parquet_files(DATABASE_PATH / "funding_rate")

if (DATABASE_PATH / "binance_futures").exists():
    explore_parquet_files(DATABASE_PATH / "binance_futures")

if (DATABASE_PATH / "binance_spot").exists():
    explore_parquet_files(DATABASE_PATH / "binance_spot")

if (DATABASE_PATH / "binance_funding_rate").exists():
    explore_parquet_files(DATABASE_PATH / "binance_funding_rate")

print("\n" + "=" * 60)
print("數據下載和整理完成！")
print("=" * 60)
print("\n✓ 所有數據已儲存為 Parquet 格式")
print("✓ 使用 pd.read_parquet() 讀取數據")
print("✓ 可使用 pd.read_parquet(path, filters=[...]) 進行快速篩選")

## 10. 使用 Parquet 數據進行共整合分析

展示如何高效地讀取和使用 Parquet 格式的數據進行進一步分析。

In [ ]:
# 示例：讀取 Binance 現貨 Parquet 數據
spot_data_dir = DATABASE_PATH / "binance_spot"
if spot_data_dir.exists():
    parquet_files = list(spot_data_dir.glob("*.parquet"))
    if parquet_files:
        symbol = parquet_files[0].name.replace(".parquet", "")
        print(f"載入 {symbol} 的現貨數據...")
        
        # 讀取整個 Parquet 文件
        df_spot = pd.read_parquet(spot_data_dir / parquet_files[0].name)
        print(f"\n完整數據形狀: {df_spot.shape}")
        print(f"\nFirst 5 rows:")
        print(df_spot.head())
        
        # 計算統計量用於共整合分析
        print(f"\n基本統計量:")
        print(df_spot[['open', 'close', 'volume']].describe())

# 示例：讀取融資費率數據
funding_dir = DATABASE_PATH / "binance_funding_rate"
if funding_dir.exists():
    parquet_files = list(funding_dir.glob("*.parquet"))
    if parquet_files:
        symbol = parquet_files[0].name.replace(".parquet", "")
        print(f"\n\n載入 {symbol} 的融資費率數據...")
        
        # 讀取融資費率 Parquet 文件
        df_funding = pd.read_parquet(funding_dir / parquet_files[0].name)
        print(f"融資費率數據形狀: {df_funding.shape}")
        print(f"\nFirst 5 rows:")
        print(df_funding.head())

print("\n\n✓ Parquet 數據讀取完成！")
print("可繼續進行共整合分析、配對交易策略等研究。")